In [1]:

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import MarkdownHeaderTextSplitter

#准备工作(离线阶段),需要有向量库存储文档的向量数据
print("====================离线阶段=====================")
#1.加载文档
with open("../resources/评估.md","r",encoding="utf-8")as f:
    md_text = f.read()
print(">md文档已经加载")
#2.切分文本
spaliter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#","h1"),("##","h2"),("###","h3")],
    strip_headers=False
)
print("> 文档结构感知切分器已就绪")
doc_list = spaliter.split_text(md_text)

#3.存入向量库
vectory_store = InMemoryVectorStore(embedding=DashScopeEmbeddings(model="text-embedding-v4"))
vectory_store.add_documents(doc_list)
print(">md文档已经存入数据库")
print("====================在线检索=====================")
#1.初始化模型
model = init_chat_model(model="deepseek-flash",extra_body = {"thinking":{"type":"disabled"}})

#2.创建智能体
system_prompt="你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，不能自己编造答案。答案要逻辑通顺，简短易懂，且要引用原文。"

def retrieve_tool(query:str)->str:
    """检索知识库，获取与query相关的知识片段"""
    #3.调用智能体
        # 根据用户的问题，检索知识库，得到相关的片.得到的结果是list[Document]。每个Document对象有metadata和page_content
    result_doc_list = vectory_store.similarity_search(query,5)
    print(f">检索知识库，得到 {len(result_doc_list)} 条结果")
    result_text_list = [result_doc.page_content for result_doc in result_doc_list]
    context = "\n".join(result_text_list)#文本片段的拼接：文本片段1\n文本片段2\n文本片段3...
    return context

agent = create_agent(model = model,tools=[retrieve_tool],system_prompt = system_prompt)
print("> 智能体初始化完成")

query = "教育的作用是什么"
print(f"> 用户提问: {query}")
    # 调用智能体得到回答
stream = agent.stream_events(
    input= {"messages":HumanMessage(content=query)},
    version = "v3"
)
for msg_stream in stream.messages:
    for msg in msg_stream.text:
        print(msg,end=" ",flush=True)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_22852\3013419937.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings


====================离线阶段=====================
>md文档已经加载
> 文档结构感知切分器已就绪
>md文档已经存入数据库
====================在线检索=====================
> 智能体初始化完成
> 用户提问: 教育的作用是什么


E:\RAG_Study\hm-insurance\agent-service\.venv\Lib\site-packages\langgraph\pregel\main.py:3805: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
E:\RAG_Study\hm-insurance\agent-service\.venv\Lib\site-packages\langgraph\pregel\main.py:3647: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


>检索知识库，得到 5 条结果
根据 知识 库 内容 ， 教育 的作用 主要体现在 教育 与社会 各 要素 的关系 中 ：

 ** 1 .   教育 对 科技 的作用 ** ： 教育 传播 科技 ； 教育 创新 科技 。

 ** 2 .   教育 对 人口 的作用 ** ： 控制 人口 的数量 ； 提高 人口 的质量 ； 改善 人口 结构 ； 调整 人才 构成 与 流动 等 作用 。

 ** 3 .   教育 对 文化 的作用 ** ： 教育 传递 和 保存 文化 ； 教育 传播 和交流 文化 ； 教育 选择和 提升 文化 ； 教育 创造 和 更新 文化 。

 此外 ， 从 教育的 本质 属性 看 ， 教育 是 " 有 目的地 培养 人的 社会 活动 "， 这也是 教育 最 根本 的作用 所在 。

 （ 注 ： 知识 库 中 教育 对社会 （ 经济 、 政治 等 ） 的作用 未 直接 涉及 ， 以上 为 可 引 用的 内容 。） 